# 02 Feature Engineering

## Goal
Transform the cleaned transaction-level dataset into a department-hour demand dataset for machine learning.

## Input
`data/cleaned_orders_products.csv`


In [2]:
import pandas as pd
import numpy as np

In [3]:
orders_products = pd.read_csv("/Users/ayu/Desktop/AI_Grocery_Demand_Forecasting/notebooks/data/cleaned_orders_products.csv")
orders_products.head()

,order_id,product_id,cart_position,is_reordered,day_of_week,hour,days_since_prior_order,product_name,aisle_id,department_id,aisle_name,dept_name
0,2,33120,1,1,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


## 1. Aggregate Demand
Create demand by day, hour, and department.

In [4]:
demand_data = (
    orders_products
    .groupby(['day_of_week', 'hour', 'dept_name'])
    .size()
    .reset_index(name='demand')
)

demand_data.head()

,day_of_week,hour,dept_name,demand
0,0,0,alcohol,52
1,0,0,babies,400
2,0,0,bakery,1301
3,0,0,beverages,2970
4,0,0,breakfast,721


In [5]:
print("Shape:", demand_data.shape)
print("Departments:", demand_data['dept_name'].nunique())

Shape: (3528, 4)
Departments: 21


## 2. Create Time Features

In [6]:
demand_data['is_weekend'] = demand_data['day_of_week'].isin([0, 6]).astype(int)

demand_data['hour_sin'] = np.sin(2 * np.pi * demand_data['hour'] / 24)
demand_data['hour_cos'] = np.cos(2 * np.pi * demand_data['hour'] / 24)

## 3. Encode Department

In [7]:
demand_data['dept_encoded'] = demand_data['dept_name'].astype('category').cat.codes

## 4. Create Log Target
Useful because demand is highly skewed.

In [9]:
demand_data['log_demand'] = np.log1p(demand_data['demand'])

In [10]:
demand_data.head()

,day_of_week,hour,dept_name,demand,is_weekend,hour_sin,hour_cos,dept_encoded,log_demand
0,0,0,alcohol,52,1,0.0,1.0,0,3.970292
1,0,0,babies,400,1,0.0,1.0,1,5.993961
2,0,0,bakery,1301,1,0.0,1.0,2,7.171657
3,0,0,beverages,2970,1,0.0,1.0,3,7.996654
4,0,0,breakfast,721,1,0.0,1.0,4,6.582025


## 5. Check Final Structure

In [11]:
print(demand_data.info())
print(demand_data.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3528 entries, 0 to 3527
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   day_of_week   3528 non-null   int64  
 1   hour          3528 non-null   int64  
 2   dept_name     3528 non-null   object 
 3   demand        3528 non-null   int64  
 4   is_weekend    3528 non-null   int64  
 5   hour_sin      3528 non-null   float64
 6   hour_cos      3528 non-null   float64
 7   dept_encoded  3528 non-null   int8   
 8   log_demand    3528 non-null   float64
dtypes: float64(3), int64(4), int8(1), object(1)
memory usage: 224.1+ KB
None
       day_of_week         hour         demand   is_weekend      hour_sin  \
count  3528.000000  3528.000000    3528.000000  3528.000000  3.528000e+03   
mean      3.000000    11.500000    9193.449263     0.285714  8.056040e-18   
std       2.000284     6.923168   18552.905882     0.451818  7.072070e-01   
min       0.000000     0.000000    

## 6. Save Processed Dataset

In [12]:
demand_data.to_csv("data/processed_demand_data.csv", index=False)
print("Processed demand dataset saved successfully.")

Processed demand dataset saved successfully.


## Summary
This notebook created:
- aggregated department-hour demand
- weekend indicator
- cyclical hour features
- department encoding
- log-transformed demand

